# 03 — Load TransLink Monthly Performance Data

Scans `~/transit-ai-data/performance/` for CSVs downloaded by the archive script,
inspects each file (shape, dtypes, nulls, date range), flags empties/failures,
and writes a `load_summary.json` for downstream notebooks.

In [1]:
# Cell 0 — Environment setup: load .env and set DATA_DIR / REPO_DIR
import os
import subprocess
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'], check=True)
    from dotenv import load_dotenv, find_dotenv

_dotenv_path = find_dotenv(usecwd=True)
if _dotenv_path:
    load_dotenv(_dotenv_path, override=False)
    print(f'Loaded .env from: {_dotenv_path}')
else:
    print('No .env file found — using defaults.')

DATA_DIR = os.environ.get('TRANSIT_AI_DATA_DIR', str(Path.home() / 'transit-ai-data'))
REPO_DIR = os.environ.get('TRANSIT_AI_REPO_DIR', str(Path.cwd().parent))

print(f'DATA_DIR: {DATA_DIR}')
print(f'REPO_DIR: {REPO_DIR}')

Loaded .env from: /Users/proteeksanyal/Desktop/Learning/Transit-AI/.env
DATA_DIR: /Users/proteeksanyal/transit-ai-data
REPO_DIR: /Users/proteeksanyal/Desktop/Learning/Transit-AI


In [2]:
# Cell 1 — List all CSVs in the performance directory
from pathlib import Path

PERF_DIR = Path(DATA_DIR) / 'performance'

if not PERF_DIR.exists():
    raise FileNotFoundError(
        f'{PERF_DIR} does not exist.\n'
        'Run scripts/archive_gtfsrt.py first to download the performance CSVs.'
    )

csv_files = sorted(PERF_DIR.glob('*.csv'))

if not csv_files:
    raise FileNotFoundError(
        f'No CSV files found in {PERF_DIR}.\n'
        'Run scripts/archive_gtfsrt.py first to download the performance CSVs.'
    )

print(f'Found {len(csv_files)} CSV file(s) in {PERF_DIR}:\n')
for f in csv_files:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<55}  {size_kb:>7.1f} KB')

Found 8 CSV file(s) in /Users/proteeksanyal/transit-ai-data/performance:

  citytrain-punctuality.csv                                    0.8 KB
  complaints.csv                                              42.9 KB
  customer-experience-ferry.csv                                1.7 KB
  customer-experience-overall.csv                              1.8 KB
  customer-experience-train.csv                                1.7 KB
  customer-experience-tram.csv                                 1.7 KB
  seq-bus-punctuality.csv                                      0.5 KB
  seq-tram-punctuality-and-reli.csv                            0.7 KB


In [3]:
# Cell 2 — Load each CSV into a DataFrame; capture failures
import pandas as pd

loaded = {}    # filename -> DataFrame
failed = {}    # filename -> error message

for path in csv_files:
    try:
        df = pd.read_csv(path, dtype=str)
        loaded[path.name] = df
    except Exception as exc:
        failed[path.name] = str(exc)

print(f'Successfully loaded : {len(loaded)}')
print(f'Failed to load      : {len(failed)}')
if failed:
    print('\nFailed files:')
    for name, err in failed.items():
        print(f'  {name}: {err}')

Successfully loaded : 8
Failed to load      : 0


In [4]:
# Cell 3 — Per-DataFrame inspection: shape, columns, dtypes, head(3), null counts, date range
import re

# Ordered from most-specific to least-specific so the right column wins first
DATE_COL_PATTERNS = [
    re.compile(r'month.?year', re.IGNORECASE),
    re.compile(r'month', re.IGNORECASE),
    re.compile(r'period', re.IGNORECASE),
    re.compile(r'date', re.IGNORECASE),
    re.compile(r'year', re.IGNORECASE),
]

def find_date_col(df):
    for pat in DATE_COL_PATTERNS:
        for col in df.columns:
            if pat.search(col):
                return col
    return None

def parse_date_range(series):
    """Try common TransLink date formats; return (min_str, max_str) or (None, None)."""
    for fmt in ('%b-%Y', '%B-%Y', '%b %Y', '%B %Y', '%m/%Y', '%Y-%m', '%Y-%m-%d', '%m-%Y'):
        try:
            parsed = pd.to_datetime(series.dropna(), format=fmt, errors='coerce').dropna()
            if len(parsed) > 0:
                return parsed.min().strftime('%Y-%m'), parsed.max().strftime('%Y-%m')
        except Exception:
            continue
    # Fallback: pandas inference
    try:
        parsed = pd.to_datetime(series.dropna(), infer_datetime_format=True, errors='coerce').dropna()
        if len(parsed) > 0:
            return parsed.min().strftime('%Y-%m'), parsed.max().strftime('%Y-%m')
    except Exception:
        pass
    return None, None

sep = '=' * 70

for name, df in loaded.items():
    print(f'\n{sep}')
    print(f'FILE: {name}')
    print(sep)

    print(f'Shape : {df.shape[0]:,} rows \u00d7 {df.shape[1]} cols')
    print(f'Columns ({df.shape[1]}): {list(df.columns)}')

    print('\nDtypes:')
    print(df.dtypes.to_string())

    print('\nhead(3):')
    print(df.head(3).to_string(index=False))

    null_counts = df.isnull().sum()
    if null_counts.any():
        print('\nNull counts (non-zero only):')
        print(null_counts[null_counts > 0].to_string())
    else:
        print('\nNull counts: none')

    date_col = find_date_col(df)
    if date_col:
        d_min, d_max = parse_date_range(df[date_col])
        if d_min:
            print(f'\nDate range ({date_col}): {d_min} \u2192 {d_max}')
        else:
            sample = list(df[date_col].dropna().unique()[:5])
            print(f'\nDate column ({date_col}): could not parse \u2014 sample values: {sample}')
    else:
        print('\nDate column: none detected')


FILE: citytrain-punctuality.csv
Shape : 28 rows × 3 cols
Columns (3): ['Month-Year', 'Citytrain: On-time running (combined peaks)', 'Citytrain: Reliability']

Dtypes:
Month-Year                                     object
Citytrain: On-time running (combined peaks)    object
Citytrain: Reliability                         object

head(3):
Month-Year Citytrain: On-time running (combined peaks) Citytrain: Reliability
  Dec-2023                                      0.9314               0.992722
  Jan-2024                                      0.9328               0.995508
  Feb-2024                                      0.9173                0.99266

Null counts: none

Date range (Month-Year): 2023-12 → 2026-03

FILE: complaints.csv
Shape : 717 rows × 6 cols
Columns (6): ['Week ending', 'Passenger trips', 'Customer complaints on \ngo card', 'Customer service complaints other than go card', 'Customer complaints\n(go card) per 10,000 trips', 'Customer complaints\n(other than go card) \nper 10,

In [5]:
# Cell 4 — Flag empty or suspicious CSVs
issues = []

for name, df in loaded.items():
    if df.empty:
        issues.append((name, 'EMPTY \u2014 zero rows'))
    elif len(df) == 1:
        issues.append((name, 'SUSPICIOUS \u2014 only 1 row (may be header-only download failure)'))
    elif df.shape[1] == 1:
        issues.append((name, 'SUSPICIOUS \u2014 only 1 column (possible parse error or wrong delimiter)'))

for name, err in failed.items():
    issues.append((name, f'LOAD FAILED \u2014 {err}'))

if issues:
    print(f'\u26a0\ufe0f  {len(issues)} issue(s) detected:\n')
    for name, reason in issues:
        print(f'  {name}: {reason}')
else:
    print(f'\u2713 All {len(loaded)} CSVs loaded cleanly \u2014 no issues detected.')

✓ All 8 CSVs loaded cleanly — no issues detected.


In [6]:
# Cell 5 — Save load_summary.json
import json

summary = {}

for name, df in loaded.items():
    date_col = find_date_col(df)
    d_min, d_max = (None, None)
    if date_col:
        d_min, d_max = parse_date_range(df[date_col])

    summary[name] = {
        'rows': len(df),
        'cols': df.shape[1],
        'date_range_start': d_min,
        'date_range_end': d_max,
    }

for name, err in failed.items():
    summary[name] = {
        'rows': None,
        'cols': None,
        'date_range_start': None,
        'date_range_end': None,
        'error': err,
    }

out_path = PERF_DIR / 'load_summary.json'
with open(out_path, 'w') as fh:
    json.dump(summary, fh, indent=2)

print(f'Saved load_summary.json -> {out_path}')
print(f'Entries: {len(summary)}')
print('\nContents:')
print(json.dumps(summary, indent=2))
print('\nNext step: run notebooks/04_eda.ipynb')

Saved load_summary.json -> /Users/proteeksanyal/transit-ai-data/performance/load_summary.json
Entries: 8

Contents:
{
  "citytrain-punctuality.csv": {
    "rows": 28,
    "cols": 3,
    "date_range_start": "2023-12",
    "date_range_end": "2026-03"
  },
  "complaints.csv": {
    "rows": 717,
    "cols": 6,
    "date_range_start": null,
    "date_range_end": null
  },
  "customer-experience-ferry.csv": {
    "rows": 24,
    "cols": 7,
    "date_range_start": null,
    "date_range_end": null
  },
  "customer-experience-overall.csv": {
    "rows": 25,
    "cols": 7,
    "date_range_start": null,
    "date_range_end": null
  },
  "customer-experience-train.csv": {
    "rows": 24,
    "cols": 7,
    "date_range_start": null,
    "date_range_end": null
  },
  "customer-experience-tram.csv": {
    "rows": 24,
    "cols": 7,
    "date_range_start": null,
    "date_range_end": null
  },
  "seq-bus-punctuality.csv": {
    "rows": 29,
    "cols": 2,
    "date_range_start": "2023-11",
    "date_ra